In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
import sys
import asyncio
from openai import AsyncOpenAI
from agents import set_default_openai_client

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

load_dotenv(override=True)

groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)
set_default_openai_client(groq_client)
print("✅ Using Groq with llama-3.3-70b-versatile")

✅ Using Groq with llama-3.3-70b-versatile


In [2]:
fetch_params = {
    "command": sys.executable,
    "args": ["-m", "mcp_server_fetch"]
}

async with MCPServerStdio(params=fetch_params, client_session_timeout_seconds=60) as server:
    fetch_tools = await server.list_tools()

fetch_tools

[Tool(name='fetch', description='Fetches a URL from the internet and returns its content as a markdown string.', input_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'fetchArguments', 'type': 'object'})]

In [3]:
import shutil
npx_path = shutil.which("npx") or shutil.which("npx.cmd")

playwright_params = {
    "command": npx_path,
    "args": ["@playwright/mcp@latest"]
}

async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as server:
    playwright_tools = await server.list_tools()

playwright_tools

[Tool(name='playwright_navigate', description='Navigate to a URL', input_schema={'properties': {'url': {'title': 'Url', 'type': 'string'}}, 'required': ['url'], 'title': 'playwright_navigateArguments', 'type': 'object'}),
 Tool(name='playwright_click', description='Click an element on the page', input_schema={'properties': {'selector': {'title': 'Selector', 'type': 'string'}}, 'required': ['selector'], 'title': 'playwright_clickArguments', 'type': 'object'}),
 Tool(name='playwright_fill', description='Fill out an input field', input_schema={'properties': {'selector': {'title': 'Selector', 'type': 'string'}, 'value': {'title': 'Value', 'type': 'string'}}, 'required': ['selector', 'value'], 'title': 'playwright_fillArguments', 'type': 'object'}),
 Tool(name='playwright_evaluate', description='Execute JavaScript in the browser', input_schema={'properties': {'script': {'title': 'Script', 'type': 'string'}}, 'required': ['script'], 'title': 'playwright_evaluateArguments', 'type': 'object'})

In [4]:
sandbox_path = os.path.abspath(os.path.join(os.getcwd(), "sandbox"))
os.makedirs(sandbox_path, exist_ok=True)

if sys.platform == "win32":
    sandbox_path = sandbox_path.replace('\\', '/')

files_params = {
    "command": npx_path,
    "args": ["-y", "@modelcontextprotocol/server-filesystem", sandbox_path]
}

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as server:
    file_tools = await server.list_tools()

file_tools

[Tool(name='read_file', description='Read the complete contents of a file', input_schema={'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'read_fileArguments', 'type': 'object'}),
 Tool(name='write_file', description='Create a new file or overwrite an existing file', input_schema={'properties': {'path': {'title': 'Path', 'type': 'string'}, 'content': {'title': 'Content', 'type': 'string'}}, 'required': ['path', 'content'], 'title': 'write_fileArguments', 'type': 'object'}),
 Tool(name='list_directory', description='Get a detailed listing of all files and directories', input_schema={'properties': {'path': {'title': 'Path', 'type': 'string'}}, 'required': ['path'], 'title': 'list_directoryArguments', 'type': 'object'})]

In [5]:
instructions = """
You browse the internet to accomplish your instructions.
You are highly capable at browsing the internet independently to accomplish your task.
When you need to write files, you do that inside the sandbox folder only.
"""

async with MCPServerStdio(params=files_params, client_session_timeout_seconds=60) as mcp_server_files:
    async with MCPServerStdio(params=playwright_params, client_session_timeout_seconds=60) as mcp_server_browser:
        agent = Agent(
            name="investigator", 
            instructions=instructions, 
            model="groq/llama-3.3-70b-versatile",
            mcp_servers=[mcp_server_files, mcp_server_browser]
        )
        with trace("investigate"):
            result = await Runner.run(agent, "Find a great recipe for Banoffee Pie, then summarize it in markdown to banoffee.md")
            print(result.final_output)

I've found a great recipe for Banoffee Pie and saved it to banoffee.md. Here's the summary:

---

# Classic Banoffee Pie Recipe

## Ingredients

### For the Base:
- 200g digestive biscuits (or graham crackers)
- 100g unsalted butter, melted
- 1 tablespoon golden syrup or honey

### For the Toffee:
- 100g unsalted butter
- 100g dark brown sugar
- 1 can (397g) sweetened condensed milk

### For the Topping:
- 3 ripe bananas
- 300ml double cream (heavy cream)
- 1 tablespoon icing sugar
- 1 teaspoon vanilla extract
- Grated chocolate or cocoa powder for dusting

## Instructions

1. **Make the Base**: Crush biscuits into fine crumbs, mix with melted butter and syrup. Press into a 20cm tart tin. Chill for 30 minutes.

2. **Make the Toffee**: Combine butter, sugar, and condensed milk in a saucepan. Heat gently until sugar dissolves, then simmer for 5-7 minutes until thickened. Pour over base. Chill for 1 hour.

3. **Assemble**: Slice bananas over the toffee. Whip cream with icing sugar and van